# Economic Institutional Ties Development

This notebook combines the completed BIT and trade-agreement panels into a bounded economic institutional-ties score. Trade flows remain a separate economic component and are not included here.

## Baseline weighting

```text
Economic institutional ties
= 0.75 × trade-agreement score
+ 0.25 × BIT score
```

Given the trade-agreement subweights, the effective maximum contributions are:

```text
Bilateral FTA:            0.4500
Regional trade agreement:             0.2625
BIT:                      0.2500
Sectoral trade agreement: 0.0375
```

This preserves the original intuition that a bilateral FTA represents roughly twice the institutional depth of a BIT or multilateral trade agreement, without using a signed net sum.

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

def locate_repository_root():
    try:
        start = Path.cwd().resolve()
    except FileNotFoundError:
        start = Path.home()
    candidates = [start]
    candidates.extend(start.parents)
    for candidate in candidates:
        if (candidate / "src" / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")

REPOSITORY_ROOT = locate_repository_root()
print("Repository root located.")

def run_script(script):
    result = subprocess.run([sys.executable, str(script)], cwd=REPOSITORY_ROOT, text=True, capture_output=True)
    if result.stdout.strip(): print(result.stdout)
    if result.stderr.strip(): print("SCRIPT STDERR:"); print(result.stderr)
    if result.returncode != 0: raise RuntimeError(f"{script.name} failed with exit code {result.returncode}")

Repository root located.


In [2]:
SCRIPT=REPOSITORY_ROOT/"src"/"data"/"build_economic_institutional_ties.py"
OUTPUT=REPOSITORY_ROOT/"data"/"processed"/"economic_institutional_ties_country_anchor_year.csv"
REPORTS=REPOSITORY_ROOT/"outputs"/"tables"/"economic_institutional_ties"
run_script(SCRIPT)

ECONOMIC INSTITUTIONAL TIES PANEL
{
  "rows": 9612,
  "countries_or_historical_entities": 199,
  "study_years": [
    2000,
    2024
  ],
  "component_weights": {
    "trade_agreement": 0.75,
    "bit": 0.25
  },
  "effective_family_weights": {
    "bilateral_fta": 0.45,
    "regional_trade_agreement": 0.2625,
    "sectoral_trade_agreement": 0.0375,
    "bit": 0.25
  },
  "bit_state_counts": {
    "none": 5345,
    "in_force": 3554,
    "signed_not_in_force": 637,
    "historical_terminated": 76
  },
  "score_distribution": {
    "count": 9612.0,
    "mean": 0.12643570536828963,
    "std": 0.16637802531232104,
    "min": 0.0,
    "25%": 0.0,
    "50%": 0.0,
    "75%": 0.25,
    "max": 0.9624999999999999
  },
  "validation_failures": {
    "duplicate_keys": 0,
    "blank_country_names": 0,
    "score_bound_failures": 0,
    "formula_identity_failures": 0,
    "missing_bit_states": 0
  }
}

Production output: data/processed/economic_institutional_ties_country_anchor_year.csv



In [3]:
panel=pd.read_csv(OUTPUT,low_memory=False)
summary=pd.read_json(REPORTS/"economic_institutional_summary.json",typ="series")
print("Rows:",len(panel)); display(summary.to_frame("value"))

Rows: 9612


,value
rows,9612
countries_or_historical_entities,199
study_years,"[2000, 2024]"
component_weights,"{'trade_agreement': 0.75, 'bit': 0.25}"
effective_family_weights,"{'bilateral_fta': 0.45, 'regional_trade_agreem..."
bit_state_counts,"{'none': 5345, 'in_force': 3554, 'signed_not_i..."
score_distribution,"{'count': 9612.0, 'mean': 0.12643570536828902,..."
validation_failures,"{'duplicate_keys': 0, 'blank_country_names': 0..."


In [4]:
keys=["country_iso3","anchor","year"]
assert len(panel)==9612
assert not panel.duplicated(keys).any()
assert panel["country_name"].notna().all()
assert panel["economic_institutional_score_raw"].between(0,1).all()
expected=0.75*panel["trade_agreement_score_raw"]+0.25*panel["bit_structural_score_raw"]
assert (panel["economic_institutional_score_raw"]-expected).abs().lt(1e-12).all()
print("PASS: keys, names, bounds, and baseline formula")

PASS: keys, names, bounds, and baseline formula


In [5]:
latest=panel[panel["year"].eq(2024)]
display(latest.groupby("anchor").agg(countries=("country_iso3","size"),mean_score=("economic_institutional_score_raw","mean"),median_score=("economic_institutional_score_raw","median"),any_trade=("any_trade_agreement_active","sum"),bit_in_force=("bit_in_force","sum")).reset_index())

,anchor,countries,mean_score,median_score,any_trade,bit_in_force
0,China,196,0.202997,0.25,28,109
1,United States,196,0.092411,0.00,21,39


In [6]:
comparison=panel.pivot(index=["country_iso3","country_name","year"],columns="anchor",values="economic_institutional_score_raw").reset_index()
comparison["us_minus_china"]=comparison["United States"]-comparison["China"]
latest_comparison=comparison[comparison["year"].eq(2024)]
print("Largest US institutional advantages")
display(latest_comparison.nlargest(15,"us_minus_china"))
print("Largest China institutional advantages")
display(latest_comparison.nsmallest(15,"us_minus_china"))

Largest US institutional advantages


anchor,country_iso3,country_name,year,China,United States,us_minus_china
3373,PAN,Panama,2024,0.0000,0.7000,0.7000
2130,JOR,Jordan,2024,0.0625,0.7000,0.6375
1805,HND,Honduras,2024,0.0000,0.5125,0.5125
449,BHR,Bahrain,2024,0.2500,0.7000,0.4500
2662,MAR,Morocco,2024,0.2500,0.7000,0.4500
3880,SLV,El Salvador,2024,0.0000,0.3250,0.3250
1200,DOM,Dominican Republic,2024,0.0000,0.2625,0.2625
1750,GTM,Guatemala,2024,0.0000,0.2625,0.2625
1725,GRD,Grenada,2024,0.0000,0.2500,0.2500
3698,RWA,Rwanda,2024,0.0000,0.2500,0.2500


Largest China institutional advantages


anchor,country_iso3,country_name,year,China,United States,us_minus_china
2255,KHM,Cambodia,2024,0.9625,0.00,-0.9625
3298,NZL,New Zealand,2024,0.9625,0.00,-0.9625
750,CHE,Switzerland,2024,0.7000,0.00,-0.7000
2030,ISL,Iceland,2024,0.7000,0.00,-0.7000
3348,PAK,Pakistan,2024,0.7000,0.00,-0.7000
224,AUS,Australia,2024,0.9625,0.45,-0.5125
2316,KOR,South Korea,2024,0.9625,0.45,-0.5125
2390,LAO,Laos,2024,0.5125,0.00,-0.5125
2900,MMR,Myanmar,2024,0.5125,0.00,-0.5125
3075,MYS,Malaya,2024,0.5125,0.00,-0.5125


In [7]:
sensitivity=(panel["economic_institutional_score_raw"]-panel["economic_institutional_equal_blocks"]).abs()
print("Mean absolute baseline/equal-block difference:",sensitivity.mean())
print("Maximum absolute difference:",sensitivity.max())
display(panel.loc[sensitivity.nlargest(20).index,["country_name","anchor","year","trade_agreement_score_raw","bit_structural_score_raw","economic_institutional_score_raw","economic_institutional_equal_blocks"]])

Mean absolute baseline/equal-block difference: 0.09505305867665417
Maximum absolute difference: 0.25


,country_name,anchor,year,trade_agreement_score_raw,bit_structural_score_raw,economic_institutional_score_raw,economic_institutional_equal_blocks
74,Angola,China,2024,0.0,1.0,0.25,0.5
100,Albania,China,2000,0.0,1.0,0.25,0.5
101,Albania,China,2001,0.0,1.0,0.25,0.5
102,Albania,China,2002,0.0,1.0,0.25,0.5
103,Albania,China,2003,0.0,1.0,0.25,0.5
104,Albania,China,2004,0.0,1.0,0.25,0.5
105,Albania,China,2005,0.0,1.0,0.25,0.5
106,Albania,China,2006,0.0,1.0,0.25,0.5
107,Albania,China,2007,0.0,1.0,0.25,0.5
108,Albania,China,2008,0.0,1.0,0.25,0.5


## Conclusion

The economic institutional score is separate from the completed trade-flow component. Component and composite thresholds remain deferred until the wider index scores are available and can be compared using substantive breaks and empirical methods including Jenks natural breaks.

# Decision register

## Baseline

- Trade-agreement score: 60% bilateral FTA, 35% regional FTA, 5% sectoral agreement.
- Economic institutional score: 75% trade-agreement score, 25% BIT score.
- Signed-only BIT: 0.25 within the BIT component.
- No categorical thresholds yet.

## Sensitivity retained

- Equal BIT/trade blocks.
- Original signed treaty orientation.
- Agreement-family and overlap counts.